[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ersilia-os/ub-cedd-projects-workshop/blob/main/sandbox/sif_feasibility_test.ipynb)

# Can an Ersilia `.sif` run in Google Colab?

Sandbox feasibility test. **Not** workshop material: it lives in `sandbox/`, is not linked
from any project README, and is not covered by the notebook rules in `CLAUDE.md`.

Run top to bottom on a **CPU** runtime. Every step prints `PASS` or `FAIL` on its own line,
so the whole output can be pasted back verbatim.

**Background.** Colab deliberately blocked the `/proc` bind mounts Apptainer needs in March 2025
([colabtools#5173](https://github.com/googlecolab/colabtools/issues/5173)); it was reported
working again in November 2025 via an `unshare -r` wrapper. This notebook settles whether it
works *today*, for a real Ersilia model image.

**The image under test** (read from its SIF header, no download required):

| | |
|---|---|
| URL | `https://models-sif.s3.eu-north-1.amazonaws.com/eos42ez_v1.sif` (public, 4.54 GB) |
| Built | 8 September 2026, Apptainer 1.4.5 |
| From | `docker://ersiliaos/eos42ez:v1.1.0` |
| Entrypoint | `sh docker-entrypoint.sh`, working dir `/root` |
| Interface | `singularity run <sif> <input.csv> <output.csv>` |

Note the `%environment` section pins `HOME=/root` when `/root/.lazyqsar` exists. Apptainer binds
the host `$HOME` by default and Colab runs as root with `HOME=/root`, which would shadow the
container's own `/root`. Every run below therefore passes **`--no-home`**.


## 1. Environment

Records the Ubuntu and kernel version, and whether unprivileged user namespaces exist at all. `max_user_namespaces: 0` means the `unshare -r` recipe is dead on arrival — stop there.


In [ ]:
%%bash
echo "=== STEP 1: environment ==="
head -2 /etc/os-release
echo "kernel: $(uname -r)"
echo "whoami: $(id -un) ($(id -u))   HOME=$HOME"
echo "max_user_namespaces: $(cat /proc/sys/user/max_user_namespaces 2>/dev/null || echo UNREADABLE)"
echo "cpus: $(nproc)   ram: $(free -g | awk '/^Mem:/{print $2}')GB   free disk: $(df -h /content | awk 'NR==2{print $4}')"

## 2. Install Apptainer

The plain `apptainer` package, not `apptainer-suid` — the user-namespace path is the one that works in a sandbox. Falls back to the upstream `.deb` if the PPA has no build for Colab's Ubuntu release.


In [ ]:
%%bash
echo "=== STEP 2: install apptainer ==="
if ! command -v apptainer >/dev/null 2>&1; then
  (add-apt-repository -y ppa:apptainer/ppa && apt-get update -qq && apt-get install -y apptainer) >/tmp/apt.log 2>&1
fi
if ! command -v apptainer >/dev/null 2>&1; then
  echo "PPA route failed; falling back to the upstream .deb"
  VER=$(curl -s https://api.github.com/repos/apptainer/apptainer/releases/latest | sed -n 's/.*"tag_name": *"v\([^"]*\)".*/\1/p')
  echo "latest release: ${VER}"
  curl -sLO "https://github.com/apptainer/apptainer/releases/download/v${VER}/apptainer_${VER}_amd64.deb"
  apt-get install -y "./apptainer_${VER}_amd64.deb" >>/tmp/apt.log 2>&1
fi
if command -v apptainer >/dev/null 2>&1; then
  apptainer --version
  echo PASS
else
  echo "FAIL - no apptainer binary; tail of the log:"
  tail -20 /tmp/apt.log
fi

## 3. Smoke test on a tiny public image

The decision point, reached before spending 4.5 GB of download. Tries the call unwrapped and then wrapped in `unshare -r`. If **neither** marker prints, containers are still blocked and the answer is no — skip to the last section.


In [ ]:
%%bash
echo "=== STEP 3: smoke test ==="
cd /content
rm -f alpine.sif
apptainer pull alpine.sif docker://alpine:latest >/tmp/pull.log 2>&1 \
  || unshare -r apptainer pull alpine.sif docker://alpine:latest >/tmp/pull.log 2>&1
if [ ! -f alpine.sif ]; then
  echo "FAIL - could not even build a SIF; tail of the log:"; tail -20 /tmp/pull.log; exit 0
fi
echo "--- plain ---"
apptainer exec /content/alpine.sif echo PLAIN_OK 2>&1 | tail -5
echo "--- unshare -r ---"
unshare -r apptainer exec /content/alpine.sif echo UNSHARE_OK 2>&1 | tail -5
echo "(PASS if either PLAIN_OK or UNSHARE_OK appeared above)"

## 4. Download the model image

The bucket is public, so this is a plain HTTPS download — no credentials. 4.54 GB, re-fetched on every fresh runtime; `-c` lets it resume if the connection drops.


In [ ]:
%%bash
echo "=== STEP 4: download the SIF ==="
cd /content
URL="https://models-sif.s3.eu-north-1.amazonaws.com/eos42ez_v1.sif"
time wget -c -q --show-progress "$URL" -O eos42ez_v1.sif
ls -lh eos42ez_v1.sif
EXPECTED=4541046784
ACTUAL=$(stat -c %s eos42ez_v1.sif)
if [ "$ACTUAL" = "$EXPECTED" ]; then
  echo "PASS (size matches)"
else
  echo "FAIL - got $ACTUAL bytes, expected $EXPECTED"
fi

## 5. Confirm the container's interface

The header already told us this is `sh docker-entrypoint.sh` with the Ersilia bundles at `/opt/ersilia`. This confirms it against the downloaded file and shows whether the lazyqsar cache is present — which is what makes `--no-home` matter.


In [ ]:
%%bash
echo "=== STEP 5: container interface ==="
apptainer inspect /content/eos42ez_v1.sif 2>&1 | head -20
echo "--- does it carry a lazyqsar cache? ---"
unshare -r apptainer exec --no-home /content/eos42ez_v1.sif \
  sh -c 'ls -d /root/.lazyqsar 2>/dev/null || echo "no lazyqsar cache"; ls /opt/ersilia' 2>&1 | tail -10

## 6. Run the model

Three familiar molecules. Tries the plausible invocations in turn — plain, `unshare -r`, then `--writable-tmpfs` in case the model writes inside the container at runtime — and reports which one worked. All of them pass `--no-home`.


In [ ]:
with open("/content/input.csv", "w") as f:
    f.write("smiles\n")
    f.write("CCO\n")                              # ethanol
    f.write("CC(=O)Oc1ccccc1C(=O)O\n")            # aspirin
    f.write("CN1C=NC2=C1C(=O)N(C)C(=O)N2C\n")     # caffeine
print(open("/content/input.csv").read())

In [ ]:
%%bash
echo "=== STEP 6: run the model ==="
cd /content
rm -f output.csv
SIF=/content/eos42ez_v1.sif
BIND="--no-home --bind /content:/content"
for ATTEMPT in "apptainer run $BIND" \
               "unshare -r apptainer run $BIND" \
               "unshare -r apptainer run --writable-tmpfs $BIND"; do
  echo "--- trying: $ATTEMPT ---"
  time $ATTEMPT "$SIF" /content/input.csv /content/output.csv 2>&1 | tail -15
  if [ -s /content/output.csv ]; then echo "PASS via: $ATTEMPT"; break; fi
done
[ -s /content/output.csv ] || echo "FAIL - no output.csv produced by any variant"

## 7. The result


In [ ]:
import os
import pandas as pd

if os.path.exists("/content/output.csv") and os.path.getsize("/content/output.csv") > 0:
    df = pd.read_csv("/content/output.csv")
    print("shape:", df.shape)
    display(df.head())
    print("VERDICT: an Ersilia SIF DOES run in Google Colab")
else:
    print("VERDICT: no output - see the failure above")

## If it failed

In descending order of how much of the container's benefit survives:

1. `apptainer build --sandbox /content/eos42ez_dir /content/eos42ez_v1.sif`, then run the
   directory instead of the SIF — sidesteps the squashfs mount entirely. Needs ~5 GB more disk.
2. Extract the squashfs payload (`apptainer sif list` for the offset, then
   `unsquashfs -o <offset>`) and run the container's conda environment at `/usr/bin/conda`
   directly from the extracted tree.
3. `proot` over that tree — userspace chroot, needs no namespaces at all.

(2) and (3) would be far too fragile to put in front of workshop participants; they are worth
trying only to characterise *why* it failed.

## If it worked

The remaining question is not technical but practical: **4.54 GB per participant per session**,
re-downloaded every time a Colab runtime is recycled. Worth measuring the wall-clock download
time in step 4 before deciding whether this belongs in a workshop notebook.
